# Stage 2 Notebook 49 - Exp2TT HybridPriorQueryHead (anchor stage 1 + K=12 query stage 2 with VFL)

**The architectural breakthrough.** NB47 (K=64 query head + Hungarian + VFL) was the FIRST experiment in 10 attempts to break the cls collapse: pos-neg gap reached 0.099 (vs 0.01 historical), val_lane_f1 hit 0.246 (vs 0.0 prior), val_lane_best_f1 hit 0.344 (vs 0.05 prior). But its geometry was weak (matched_iou=0.27 vs anchor's 0.54 in NB48) because K=64 free-form queries can't cover lane space as densely as 192 geometrically-initialized priors.

Meanwhile NB48 (anchor + VFL + full 70K data) hit project geometry record (matched_iou=0.544, oracle_f1=0.467) but its cls stayed at the dynamic-k matching collapse (gap=0.015).

**The two heads have orthogonal strengths.** Exp2TT combines them via the existing `HybridPriorQueryHead`:

- **Stage 1**: 192 anchor priors with dynamic-k matching, supervised by `stage1_aux_loss_weight=1.0`. Inherits NB48's geometry (matched_iou=0.54).
- **Stage 2**: K=12 learned queries that cross-attend to stage 1's per-prior features AND the spatial feature map. Hungarian 1-to-1 matching + VFL. Inherits NB47's clean cls signal.
- Inference uses stage 2's cls scores to rank, with stage 2's curves (which started from stage 1's geometry-rich feature pool) for the geometry.

Reference: this is the `HybridPriorQueryHead` from NB22 (Exp2Q). NB22 failed because stage 1 had no direct supervision back then; we now have `stage1_aux_loss_weight` infrastructure (added for Exp2T) so stage 1 trains exactly like NB48's anchor head.

Reference: combines DETR-style query supervision with anchor-style geometric priors, conceptually similar to Deformable DETR's reference points or DAB-DETR's anchor queries.

### Run mode

1. `DEBUG_MODE = True` smoke first.
2. `DEBUG_MODE = False` for 20 epochs at limit=3000.
3. AMP keeps wall-clock ~ 30 minutes for 20 epochs at 3000 samples.
4. Independent of all prior NBs.

In [ ]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

In [ ]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp44_rmt_gca_hybrid_anchor_query_vfl_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

In [ ]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp44_rmt_gca_hybrid_anchor_query_vfl_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short20'
    EPOCHS = 20
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

## What to watch in Exp2TT training

Reference NB47 (K=64 query alone): pos-neg gap=0.099, val_lane_f1=0.246, matched_iou=0.27, decoded_f1=0.019.
Reference NB48 (anchor + full data): pos-neg gap=0.015, val_lane_f1=0.07, matched_iou=0.544, decoded_f1=0.05.

Pass criteria at epoch 20:
- **`val/matched_line_iou >= 0.45`** -- inherit NB48-style geometry from stage 1 (with limit=3000, expect slightly below 0.544; 0.45 is a reasonable lower bound).
- **`pos_score - neg_score >= 0.10`** on stage 2 -- inherit NB47-style cls discrimination via Hungarian.
- **`val/lane_f1 >= 0.20` and `val/lane_best_f1 >= 0.30`** -- decisive evidence the cls is now ranking real lanes.
- **`val/lane/decoded_f1 >= 0.10`** -- 2x NB48's 0.05 because cls is finally discriminative.
- `lane/stage1_aux_total` should DECREASE monotonically -- confirms stage 1 is being supervised.

Failure signals:
- matched_iou < 0.30: hybrid stage 2 distorts geometry. Set `stage1_aux_loss_weight=2.0`, `aux_stage_loss_weight=0.0`.
- pos-neg gap < 0.05: K=12 queries are too few; switch to K=24 in a follow-up.
- val_det >> 2.5: hybrid head adds enough capacity that joint conflict resurges. Drop lambda_lane to 0.7.